Cargamos las librierias necesarias para el proyecto

In [ ]:
import tensorflow as tf
import numpy as np
from tensorflow import keras
from google.colab import drive
import shutil
import cv2
import pandas as pd
import matplotlib.pyplot as plt
import os

Cargamos los datos desde drive.

In [ ]:
drive.mount('/content/drive/', force_remount=False)
shutil.copy("/content/drive/MyDrive/kaggle.zip","/content/")
!unzip kaggle.zip
#shutil.copy("/content/drive/MyDrive/InfoEspecies.csv","/content/")

Cargamos los datos en python y arreglamos las rutas.

In [ ]:
def ArreglarLasRutas(df):
    rutas_nuevas = []
    for path in df.image_path:
        path = path.lstrip("/")
        rutas_nuevas.append(path)
    df.image_path = rutas_nuevas


def muestras_data_set(df,n, seed, use_seed = False):
    especies = pd.unique(df.label)
    df_final = pd.DataFrame(columns=df.columns)
    for especie in especies:
        df_fil = df[df.label == especie]
        if use_seed:
            df_fil = df_fil.sample(n=n, random_state=seed)
        else:
            df_fil = df_fil.sample(n=n)
        df_final = pd.concat([df_final, df_fil], ignore_index=True)
    return df_final

In [ ]:
rutas_val = pd.read_csv("kaggle/working/val.csv")
rutas_train = pd.read_csv("kaggle/working/train.csv")
rutas_test = pd.read_csv("kaggle/working/test.csv")
info_especies = pd.read_csv("InfoEspecies.csv")
info_especies = info_especies.drop(info_especies.columns[0], axis= "columns")

#rutas_train = muestras_data_set(df = rutas_train,n = 50, seed = 666, use_seed= True)
#rutas_test = muestras_data_set(df = rutas_test,n = 5, seed = 666, use_seed= True)
#rutas_val = muestras_data_set(df = rutas_val,n = 5, seed = 666, use_seed= True)

train = pd.merge(info_especies, rutas_train, on='label', how='inner')
test = pd.merge(info_especies, rutas_test, on='label', how='inner')
val = pd.merge(info_especies, rutas_val, on='label', how='inner')


dfs = [train, val, test]
for df in dfs:
    ArreglarLasRutas(df)

In [ ]:
def Modificar_direc(df_rutas, subset):
    os.makedirs(f"Data/{subset}", exist_ok=True)

    ArreglarLasRutas(df_rutas)


    for path, label in zip(df_rutas.image_path, df_rutas.label):
        destino = f"Data/{subset}/{label}/"
        os.makedirs(destino, exist_ok=True)
        print(f"Copiando {path} --> {destino}")
        shutil.copy(path, destino)
        print(f"Terminado {label}")

In [ ]:
Modificar_direc(train, "train")

In [ ]:
Modificar_direc(val, "val")

In [ ]:
IMG_SIZE = (640, 480)
BATCH_SIZE = 64

train_dataset = tf.keras.utils.image_dataset_from_directory(
    "Data/train",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical"
)

val_dataset = tf.keras.utils.image_dataset_from_directory(
    "Data/val",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical"
)



In [ ]:
from tensorflow.keras.applications.efficientnet_v2 import preprocess_input

train_dataset = train_dataset.map(
    lambda x, y: (preprocess_input(x), y)
)

val_dataset = val_dataset.map(
    lambda x, y: (preprocess_input(x), y)
)

In [ ]:
from keras.applications import EfficientNetV2L

base_model = EfficientNetV2L(weights='imagenet', include_top=False, input_shape=(640, 480, 3))
#base_model.summary()



In [ ]:
from keras import Model
from keras.layers import Input, Dense, GlobalAveragePooling2D
x = base_model.output
x = GlobalAveragePooling2D()(x)
predictions = Dense(54, activation='softmax')(x)
model = Model(inputs=base_model.input, outputs=predictions)

In [ ]:
for layer in base_model.layers:
    layer.trainable = False

# Compilamos
model.compile(optimizer='Adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

In [ ]:
hist = model.fit(train_dataset, validation_data = val_dataset, epochs=6)

In [ ]:
loss = hist.history['loss']
val_loss = hist.history["val_loss"]

plt.plot(loss, label = "loss")
plt.plot(val_loss, label = "val_loss")
plt.legend()
plt.show()

In [ ]:
acc = hist.history['accuracy']
val_acc = hist.history["val_accuracy"]

plt.plot(acc, label = "acc")
plt.plot(val_acc, label = "val_acc")
plt.legend()
plt.show()

In [ ]:
model.save("EffTune.keras")

In [ ]:
shutil.copy(
    "EffTune.keras",
    "/content/drive/MyDrive/")

In [ ]:
df_hist = pd.DataFrame(hist.history)
df_hist.to_csv("effresult.csv", index=False)
shutil.copy(
    "effresult.csv",
    "/content/drive/MyDrive/")

# Continuar entrenamiento

En este apartado volveremos a entrenar el modelo.Primero cargaremos los resultados anteriores, para llevar el registro

In [ ]:
df_hist = pd.read_csv("effresult.csv")
df_hist.head()

Ahora cargamos el modelo.

In [ ]:
model = keras.models.load_model("EffTune.keras")

Y por ultimo compilamos y entrenamos.

In [ ]:
model.compile(optimizer='Adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

In [ ]:
hist = model.fit(train_dataset, validation_data = val_dataset, epochs=4)

In [ ]:
df_hist_new = pd.DataFrame(hist.history)
df_hist_new = pd.concat([df_hist, df_hist_new], ignore_index=True)
df_hist_new

In [ ]:
loss = df_hist_new['loss']
val_loss = df_hist_new["val_loss"]

plt.plot(loss, label = "loss")
plt.plot(val_loss, label = "val_loss")
plt.legend()
plt.show()

In [ ]:
acc = df_hist_new['accuracy']
val_acc = df_hist_new["val_accuracy"]

plt.plot(acc, label = "acc")
plt.plot(val_acc, label = "val_acc")
plt.legend()
plt.show()

In [ ]:
model.save("EffTune.keras")

In [ ]:
shutil.copy(
    "EffTune.keras",
    "/content/drive/MyDrive/")

In [ ]:
df_hist_new.to_csv("effresult.csv", index=False)
shutil.copy(
    "effresult.csv",
    "/content/drive/MyDrive/")